In [11]:
import pandas as pd

## Load Data

In [12]:
# Load datasets
df_income = pd.read_csv('../data/raw/income_district.csv')
df_poverty = pd.read_csv('../data/raw/poverty_district.csv')
df_gini = pd.read_csv('../data/raw/gini_district.csv')
df_amenities = pd.read_csv('../data/raw/amenities.csv')

# Standardize column names
df_income.columns = df_income.columns.str.strip().str.lower()
df_poverty.columns = df_poverty.columns.str.strip().str.lower()
df_gini.columns = df_gini.columns.str.strip().str.lower()
df_amenities.columns = df_amenities.columns.str.strip().str.lower()

# Convert date to datetime and extract year
df_income['date'] = pd.to_datetime(df_income['date'])
df_income['year'] = df_income['date'].dt.year

df_poverty['date'] = pd.to_datetime(df_poverty['date'])
df_poverty['year'] = df_poverty['date'].dt.year

df_gini['date'] = pd.to_datetime(df_gini['date'])
df_gini['year'] = df_gini['date'].dt.year

df_amenities['date'] = pd.to_datetime(df_amenities['date'])
df_amenities['year'] = df_amenities['date'].dt.year

## Filter Data to Years 2019 and 2022

In [13]:
# Filter all DataFrames to keep only years 2019 or 2022
df_income_filtered = df_income[df_income['year'].isin([2019, 2022])].copy()
df_poverty_filtered = df_poverty[df_poverty['year'].isin([2019, 2022])].copy()
df_gini_filtered = df_gini[df_gini['year'].isin([2019, 2022])].copy()
df_amenities_filtered = df_amenities[df_amenities['year'].isin([2019, 2022])].copy()

print(f"Income filtered shape: {df_income_filtered.shape}")
print(f"Poverty filtered shape: {df_poverty_filtered.shape}")
print(f"Gini filtered shape: {df_gini_filtered.shape}")
print(f"Amenities filtered shape: {df_amenities_filtered.shape}")

Income filtered shape: (318, 6)
Poverty filtered shape: (318, 6)
Gini filtered shape: (318, 5)
Amenities filtered shape: (340, 7)


## Merge DataFrames Using Left Joins

In [14]:
# Start with df_income as the base
df_merged = df_income_filtered.copy()

# Left-join df_poverty
df_merged = df_merged.merge(
    df_poverty_filtered[['state', 'district', 'year', 'poverty_absolute', 'poverty_relative']],
    on=['state', 'district', 'year'],
    how='left'
)

# Left-join df_gini
df_merged = df_merged.merge(
    df_gini_filtered[['state', 'district', 'year', 'gini']],
    on=['state', 'district', 'year'],
    how='left'
)

# Left-join df_amenities
df_merged = df_merged.merge(
    df_amenities_filtered[['state', 'district', 'year', 'piped_water', 'sanitation', 'electricity']],
    on=['state', 'district', 'year'],
    how='left'
)

## Inspect Merged DataFrame

In [15]:
# Print the shape of the merged DataFrame
print(f"Merged DataFrame shape: {df_merged.shape}")

Merged DataFrame shape: (318, 12)


In [16]:
# Print the column names
print(f"\nColumn names:\n{df_merged.columns.tolist()}")


Column names:
['state', 'district', 'date', 'income_mean', 'income_median', 'year', 'poverty_absolute', 'poverty_relative', 'gini', 'piped_water', 'sanitation', 'electricity']


In [17]:
# Print the number of missing values per column
print(f"\nMissing values per column:")
print(df_merged.isnull().sum())
print(df_merged)


Missing values per column:
state               0
district            0
date                0
income_mean         0
income_median       0
year                0
poverty_absolute    0
poverty_relative    0
gini                0
piped_water         7
sanitation          6
electricity         6
dtype: int64
                 state           district       date  income_mean  \
0                Johor         Batu Pahat 2019-01-01         7392   
1                Johor         Batu Pahat 2022-01-01         7419   
2                Johor        Johor Bahru 2019-01-01         9315   
3                Johor        Johor Bahru 2022-01-01         9869   
4                Johor             Kluang 2019-01-01         5953   
..                 ...                ...        ...          ...   
313  W.P. Kuala Lumpur  W.P. Kuala Lumpur 2022-01-01        13325   
314        W.P. Labuan        W.P. Labuan 2019-01-01         8319   
315        W.P. Labuan        W.P. Labuan 2022-01-01         8250   
316  

## Observation Counts by Year and Level

**Note:** Two cross-sectional observations per geographic unit.

In [18]:
# Create observation count table by Year and Level (District)
obs_counts = df_merged.groupby(['year']).size().reset_index(name='Observations')
obs_counts['Level'] = 'District'
obs_counts = obs_counts[['year', 'Level', 'Observations']]
obs_counts.columns = ['Year', 'Level', 'Observations']

# Display the table
print("Observation Counts by Year and Level")
print("=" * 40)
print(obs_counts.to_string(index=False))
print("=" * 40)
print(f"\nTotal observations: {obs_counts['Observations'].sum()}")
print(f"Number of districts: {df_merged['district'].nunique()}")

Observation Counts by Year and Level
 Year    Level  Observations
 2019 District           158
 2022 District           160

Total observations: 318
Number of districts: 160


In [19]:
# Clean display of observation counts table
obs_counts

,Year,Level,Observations
0,2019,District,158
1,2022,District,160


In [20]:
df_merged.head(10)

,state,district,date,income_mean,income_median,year,poverty_absolute,poverty_relative,gini,piped_water,sanitation,electricity
0,Johor,Batu Pahat,2019-01-01,7392,6504,2019,2.9,9.0,0.295,100.0,100.00,100.0
1,Johor,Batu Pahat,2022-01-01,7419,6347,2022,5.1,19.4,0.338,100.0,100.00,100.0
2,Johor,Johor Bahru,2019-01-01,9315,7342,2019,3.3,12.8,0.388,100.0,100.00,100.0
3,Johor,Johor Bahru,2022-01-01,9869,8232,2022,3.7,10.4,0.359,100.0,100.00,100.0
4,Johor,Kluang,2019-01-01,5953,4933,2019,5.0,24.9,0.333,100.0,100.00,100.0
5,Johor,Kluang,2022-01-01,6461,5204,2022,7.2,27.4,0.354,99.7,100.00,100.0
6,Johor,Kota Tinggi,2019-01-01,6982,5475,2019,6.0,20.8,0.361,98.8,98.98,100.0
7,Johor,Kota Tinggi,2022-01-01,7529,6227,2022,5.0,17.0,0.343,100.0,99.85,100.0
8,Johor,Kulai,2019-01-01,8602,7536,2019,3.2,10.1,0.324,100.0,100.00,100.0
9,Johor,Kulai,2022-01-01,9177,7460,2022,0.4,7.4,0.337,100.0,100.00,100.0
